# Spark Assignment: NYC Yellow Taxi 2024 Analysis

**Student**: Oscar Romero López.

**Data**: NYC Yellow Taxi trip records from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

---

## Assignment Overview

This notebook demonstrates the complete Spark data analysis pipeline across 5 parts:
- **Part 1** (15%): Spark RDDs
- **Part 2** (25%): DataFrames: Cleaning and Transforming
- **Part 3** (35%): Spark SQL & Window functions
- **Part 4** (10%): Performance Analysis
- **Part 5** (15%): Final Summary Report

The notebook has been executed in Google Colab.

## Setup

Prerequisites and configuration.

### Setup: Install Packages

Ensure dependencies are available.

In [26]:
print("Installing required packages...\n")

import subprocess
import sys

packages = ["wget"]

for package in packages:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        print(f" {package} installed")
    except:
        print(f"! {package} may already be installed")

print(f"\n Packages ready")

Installing required packages...

 wget installed

 Packages ready


### Setup: Configure Spark

Initialize Spark session with optimized configurations.

In [27]:
print("Configuring Spark session...\n")

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pyspark

print(f"Spark version: {pyspark.__version__}")

spark = SparkSession.builder \
    .appName("Assignment2") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

print(f"Driver memory: {spark._jsc.sc().getConf().get('spark.driver.memory')}")
print(f"Shuffle partitions: {spark._jsc.sc().getConf().get('spark.sql.shuffle.partitions')}")

sc = spark.sparkContext
print(f"Master: {sc.master}")
print(f"App name: {sc.appName}")
print(f"\n Spark session configured")

Configuring Spark session...

Spark version: 4.0.2
Driver memory: 4g
Shuffle partitions: 8
Master: local[*]
App name: Assignment2

 Spark session configured


### Setup: Download Data

Download NYC Yellow Taxi data for 2024 (all 12 months).

In [28]:
print("Downloading NYC Yellow Taxi 2024 data (12 months)...")

import subprocess
import os

os.makedirs("/app/data", exist_ok=True)

for month in range(1, 13):
    filename = f"yellow_tripdata_2024-{month:02d}.parquet"
    filepath = f"/app/data/{filename}"

    if os.path.exists(filepath):
        print(f"✓ {filename} already exists")
    else:
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"
        print(f"Downloading {filename}...")
        try:
            subprocess.run(["wget", "-q", url, "-O", filepath], check=True, timeout=300)
            file_size = os.path.getsize(filepath) / (1024 * 1024)  # Size in MB
            print(f"  Downloaded ({file_size:.1f} MB)")
        except Exception as e:
            print(f"  Error: {e}")

✓ yellow_tripdata_2024-01.parquet already exists
✓ yellow_tripdata_2024-02.parquet already exists
✓ yellow_tripdata_2024-03.parquet already exists
✓ yellow_tripdata_2024-04.parquet already exists
✓ yellow_tripdata_2024-05.parquet already exists
✓ yellow_tripdata_2024-06.parquet already exists
✓ yellow_tripdata_2024-07.parquet already exists
✓ yellow_tripdata_2024-08.parquet already exists
✓ yellow_tripdata_2024-09.parquet already exists
✓ yellow_tripdata_2024-10.parquet already exists
✓ yellow_tripdata_2024-11.parquet already exists
✓ yellow_tripdata_2024-12.parquet already exists


## Part 1: Spark RDDs - Low-Level Operations (15%)

RDDs (Resilient Distributed Datasets) are the fundamental abstraction in Spark.

### 1.1 Load as RDD

In [29]:
print("=== 1.1 Load as RDD ===\n")

# Load January data as DataFrame
df_january = spark.read.parquet("/app/data/yellow_tripdata_2024-01.parquet")

# Convert to RDD
rdd_january = df_january.rdd

print("First 3 rows from RDD (as Row objects):")
for row in rdd_january.take(3):
    print(f"  {row}")

=== 1.1 Load as RDD ===

First 3 rows from RDD (as Row objects):
  Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0)
  Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 3), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 0, 9, 36), passenger_count=1, trip_distance=1.8, RatecodeID=1, store_and_fwd_flag='N', PULocationID=140, DOLocationID=236, payment_type=1, fare_amount=10.0, extra=3.5, mta_tax=0.5, tip_amount=3.75, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=18.75, congestion_surcharge=2.5, Airport_fee=0.0)
  Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 

In [30]:
first_row = rdd_january.first()
print(f"Type of an RDD element: {type(first_row)}")

Type of an RDD element: <class 'pyspark.sql.types.Row'>


### 1.2 RDD Transformations

In [31]:
print("=== 1.2 RDD Transformations ===\n")

# 1.2.1 Count trips with high passenger count


=== 1.2 RDD Transformations ===



In [32]:
high_passenger_trips = rdd_january.filter(lambda row: row['passenger_count'] > 4).count()
print(f"   Trips with >4 passengers: {high_passenger_trips:,}")

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 206.0 failed 1 times, most recent failure: Lost task 6.0 in stage 206.0 (TID 675) (da8ef1430e52 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2044, in main
    process()
  File "/usr/local/lib/python3.12/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2034, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/core/rdd.py", line 5306, in pipeline_func
    return func(split, prev_func(split, iterator))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/core/rdd.py", line 5306

Null problems, let's take it into account.

In [33]:
# 1.2.1 Count trips with high passenger count (with null safety)
print(f"   Total trips: {rdd_january.count():,}")

# 1.2.2 Count trips with 'passenger_count'>4

# Added a check: row['passenger_count'] is not None
high_passenger_trips = rdd_january.filter(
    lambda row: row['passenger_count'] is not None and row['passenger_count'] > 4
).count()

print(f"   Trips with >4 passengers: {high_passenger_trips:,}")

   Total trips: 2,964,624
   Trips with >4 passengers: 55,919


In [34]:
# 1.2.3 Average trip distance
# Filter first to ensure we only have numbers
distances = rdd_january \
    .filter(lambda row: row['trip_distance'] is not None) \
    .map(lambda row: (float(row['trip_distance']), 1))

# Now reduce is safe
sum_count = distances.reduce(lambda x, y: (x[0] + y[0], x[1] + y[1]))
avg_distance = sum_count[0] / sum_count[1] if sum_count[1] > 0 else 0

print(f"   Average distance: {avg_distance:.2f} miles")

   Average distance: 3.65 miles


### 1.3 WordCount-style Aggregation

In [35]:
print("=== 1.3 Aggregation ===\n")

# 1.3.1 Total revenue per pickup location
print("Top 5 locations by total revenue (January):")
location_revenue = rdd_january.map(lambda row: (row['PULocationID'], row['total_amount']))

revenue_by_location = location_revenue.reduceByKey(lambda x, y: x + y)

# 1.3.2. Top 5 pickup locations by revenue
top_5 = revenue_by_location.sortBy(lambda x: x[1], ascending=False).take(5)

for location_id, revenue in top_5:
    print(f"   Location {location_id}: ${revenue:,.2f}")

print(f"\nTotal unique locations: {revenue_by_location.count()}")
print(f"Total revenue: ${revenue_by_location.map(lambda x: x[1]).reduce(lambda x, y: x + y):,.2f}")

=== 1.3 Aggregation ===

Top 5 locations by total revenue (January):
   Location 132: $11,121,928.53
   Location 138: $5,820,969.96
   Location 161: $3,369,045.01
   Location 230: $2,793,051.68
   Location 237: $2,776,195.32

Total unique locations: 260
Total revenue: $79,456,384.28


### 1.4 Reflection: RDDs vs DataFrames

1. **Readability**: RDDs require detailed functional programming (map/filter/reduce). DataFrames use familiar SQL-like operations that are more intuitive.

2. **Performance**: Catalyst optimizer analyzes DataFrame operations and generates optimized physical plans. RDDs bypass this optimization layer.

3. **Code comparison**:
   - RDD: `rdd.map(lambda row: (row[8], row[16])).reduceByKey(lambda x, y: x + y)`
   - DataFrame: `df.groupBy("PULocationID").agg(sum("total_amount"))`

4. **Default Behavior**: DataFrames provide intelligent caching, automatic schema inference, and better memory management.

**Conclusion**: For structured data (tabular with named columns), always use DataFrames/SQL over RDDs unless you have unstructured data or require very low-level distributed operations.

## Part 2: DataFrames - Cleaning and Transforming (25%)

Now switching to DataFrames for the full year of data.

### 2.1 Load and Combine


In [36]:
print("=== 2.1 Load and Combine ===\n")

# Load all 12 months using glob pattern
df_full = spark.read.parquet("/app/data/yellow_tripdata_2024-*.parquet")

print("Schema:")
df_full.printSchema()

row_count = df_full.count()
print(f"\nTotal rows: {row_count:,}")
print(f"Total columns: {len(df_full.columns)}")

=== 2.1 Load and Combine ===

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)


Total rows: 41,169,720
Total columns: 19


### 2.2 Data Exploration


In [37]:
print("=== 2.2 Data Exploration ===\n")

# 2.2.1 Dimensions
print("1. Dataset dimensions:")
print(f"   Rows: {row_count:,}")
print(f"   Columns: {len(df_full.columns)}")

# 2.2.2 Null values
print("\n2. Null values per column:")
null_counts = df_full.select([count(when(col(c).isNull(), 1)).alias(c) for c in df_full.columns])
null_results = null_counts.collect()[0].asDict()
for col_name, null_count in sorted(null_results.items(), key=lambda x: x[1], reverse=True):
    if null_count > 0:
        percentage = (null_count / row_count) * 100
        print(f"   {col_name}: {null_count:,} ({percentage:.2f}%)")

# 2.2.3 Date range
print("\n3. Date range (tpep_pickup_datetime):")
date_range = df_full.agg(
    min("tpep_pickup_datetime").alias("min_date"),
    max("tpep_pickup_datetime").alias("max_date")
).collect()[0]
print(f"   Min: {date_range['min_date']}")
print(f"   Max: {date_range['max_date']}")

# 2.2.4 Payment types
print("\n4. Payment types and trip counts:")
payment_summary = df_full.groupBy("payment_type").count().orderBy(desc("count"))
payment_summary.show()

=== 2.2 Data Exploration ===

1. Dataset dimensions:
   Rows: 41,169,720
   Columns: 19

2. Null values per column:
   passenger_count: 4,091,232 (9.94%)
   RatecodeID: 4,091,232 (9.94%)
   store_and_fwd_flag: 4,091,232 (9.94%)
   congestion_surcharge: 4,091,232 (9.94%)
   Airport_fee: 4,091,232 (9.94%)

3. Date range (tpep_pickup_datetime):
   Min: 2002-12-31 16:46:07
   Max: 2026-06-26 23:53:12

4. Payment types and trip counts:
+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|30452159|
|           2| 5540088|
|           0| 4091232|
|           4|  794494|
|           3|  291743|
|           5|       4|
+------------+--------+



### 2.3 Data Cleaning

In [38]:
print("=== 2.3 Data Cleaning ===\n")

df_clean = df_full
initial_count = df_clean.count()
print(f"Initial row count: {initial_count:,}")

# Step 1: Remove rows where trip_distance <= 0 or total_amount <= 0
step1_count = df_clean.count()
df_clean = df_clean.filter((col("trip_distance") > 0) & (col("total_amount") > 0))
removed_1 = step1_count - df_clean.count()
print(f"\n1. Remove invalid trips (distance ≤ 0 or amount ≤ 0):")
print(f"   Rows removed: {removed_1:,} ({removed_1/step1_count*100:.2f}%)")

# Step 2: Remove rows where passenger_count is null or 0
step2_count = df_clean.count()
df_clean = df_clean.filter((col("passenger_count").isNotNull()) & (col("passenger_count") > 0))
removed_2 = step2_count - df_clean.count()
print(f"\n2. Remove null or zero passenger_count:")
print(f"   Rows removed: {removed_2:,} ({removed_2/step2_count*100:.2f}%)")

# Step 3: Remove outliers - distance > 200 miles or amount > $5000
step3_count = df_clean.count()
df_clean = df_clean.filter((col("trip_distance") <= 200) & (col("total_amount") <= 5000))
removed_3 = step3_count - df_clean.count()
print(f"\n3. Remove distance/amount outliers (distance > 200 or amount > $5000):")
print(f"   Rows removed: {removed_3:,} ({removed_3/step3_count*100:.2f}%)")

# Step 4: Remove time anomalies - pickup date outside 2024
step4_count = df_clean.count()
df_clean = df_clean.filter(
    (year("tpep_pickup_datetime") == 2024)
)
removed_4 = step4_count - df_clean.count()
print(f"\n4. Remove time anomalies (pickup date not in 2024):")
print(f"   Rows removed: {removed_4:,} ({removed_4/step4_count*100:.2f}%)")

# Summary
total_removed = initial_count - df_clean.count()
print(f"\n" + "="*50)
print(f"Total rows removed: {total_removed:,}")
print(f"Total percentage cleaned: {total_removed/initial_count*100:.2f}%")
print(f"Rows remaining: {df_clean.count():,} ({df_clean.count()/initial_count*100:.2f}%)")

=== 2.3 Data Cleaning ===

Initial row count: 41,169,720

1. Remove invalid trips (distance ≤ 0 or amount ≤ 0):
   Rows removed: 1,336,887 (3.25%)

2. Remove null or zero passenger_count:
   Rows removed: 4,200,008 (10.54%)

3. Remove distance/amount outliers (distance > 200 or amount > $5000):
   Rows removed: 268 (0.00%)

4. Remove time anomalies (pickup date not in 2024):
   Rows removed: 55 (0.00%)

Total rows removed: 5,537,218
Total percentage cleaned: 13.45%
Rows remaining: 35,632,502 (86.55%)


### 2.4 Feature Engineering

In [39]:
print("=== 2.4 Feature Engineering ===\n")

from pyspark.sql.functions import month, hour, dayofweek, col, coalesce, when, unix_timestamp, abs as spark_abs

df_clean = df_clean.withColumns({
    # 1. Trip duration in minutes
    "trip_duration_minutes": (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,

    # 2. Hour of day (0-23)
    "hour_of_day": hour("tpep_pickup_datetime"),

    # 3. Day of week (1=Monday, 7=Sunday)
    "day_of_week": dayofweek("tpep_pickup_datetime"),

    # 4. Month
    "month": month("tpep_pickup_datetime"),
})

# 5. Tip percentage (handle division by zero)
df_clean = df_clean.withColumn(
    "tip_percentage",
    when(col("fare_amount") != 0, (col("tip_amount") / col("fare_amount")) * 100).otherwise(0)
)

# 6. Speed in mph (handle division by zero and filter unreasonable speeds)
df_clean = df_clean.withColumn(
    "speed_mph",
    when(
        (col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") / 60 > 0),
        col("trip_distance") / (col("trip_duration_minutes") / 60)
    ).otherwise(0)
)

# Filter out unreasonable speeds (> 100 mph)
df_clean = df_clean.filter(col("speed_mph") <= 100)

print("New columns added:")
print("  1. trip_duration_minutes")
print("  2. hour_of_day")
print("  3. day_of_week")
print("  4. month")
print("  5. tip_percentage")
print("  6. speed_mph (filtered out > 100 mph)")

print(f"\nRows after filtering high speeds: {df_clean.count():,}")

# Display sample
print("\nSample enriched data:")
df_clean.select(
    "tpep_pickup_datetime",
    "trip_distance",
    "total_amount",
    "trip_duration_minutes",
    "hour_of_day",
    "tip_percentage",
    "speed_mph"
).limit(5).show(truncate=False)

=== 2.4 Feature Engineering ===

New columns added:
  1. trip_duration_minutes
  2. hour_of_day
  3. day_of_week
  4. month
  5. tip_percentage
  6. speed_mph (filtered out > 100 mph)

Rows after filtering high speeds: 35,623,564

Sample enriched data:
+--------------------+-------------+------------+---------------------+-----------+------------------+------------------+
|tpep_pickup_datetime|trip_distance|total_amount|trip_duration_minutes|hour_of_day|tip_percentage    |speed_mph         |
+--------------------+-------------+------------+---------------------+-----------+------------------+------------------+
|2024-10-01 00:30:44 |3.0          |24.9        |17.7                 |0          |8.152173913043478 |10.16949152542373 |
|2024-10-01 00:12:20 |2.2          |23.0        |13.083333333333334   |0          |26.76056338028169 |10.089171974522294|
|2024-10-01 00:04:46 |2.7          |22.2        |9.1                  |0          |27.40740740740741 |17.802197802197803|
|2024-10-01 00:

### 2.5 Aggregations

In [40]:
print("=== 2.5 Aggregations ===\n")

# 1. Average trip distance and fare per hour of day
print("1. Average trip distance and fare by hour of day:")
hourly_stats = df_clean.groupBy("hour_of_day").agg(
    avg("trip_distance").alias("avg_distance"),
    avg("fare_amount").alias("avg_fare")
).orderBy("hour_of_day")

hourly_stats.show(24, truncate=False)

max_distance_hour = hourly_stats.orderBy(desc("avg_distance")).first()
max_fare_hour = hourly_stats.orderBy(desc("avg_fare")).first()
print(f"   Longest trips: Hour {max_distance_hour['hour_of_day']} ({max_distance_hour['avg_distance']:.2f} miles)")
print(f"   Most expensive: Hour {max_fare_hour['hour_of_day']} (${max_fare_hour['avg_fare']:.2f})")

# 2. Total revenue per month
print("\n2. Total revenue per month:")
monthly_revenue = df_clean.groupBy("month").agg(
    sum("total_amount").alias("total_revenue"),
    count("*").alias("trip_count")
).orderBy("month")

monthly_revenue.show(12, truncate=False)

seasonal_pattern = monthly_revenue.collect()
min_month = __builtins__.min(seasonal_pattern, key=lambda x: x['total_revenue'])
max_month =__builtins__. max(seasonal_pattern, key=lambda x: x['total_revenue'])
print(f"   Lowest revenue: Month {min_month['month']} (${min_month['total_revenue']:,.0f})")
print(f"   Highest revenue: Month {max_month['month']} (${max_month['total_revenue']:,.0f})")
print(f"   We find an increasing in spring and by the end of the year")

# 3. Trip count per day of week
print("\n3. Trip count per day of week:")
day_names = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 7: "Sunday"}
daily_trips = df_clean.groupBy("day_of_week").agg(
    count("*").alias("trip_count")
).orderBy("day_of_week")

daily_results = daily_trips.collect()
for row in daily_results:
    day_name = day_names[row['day_of_week']]
    print(f"   {day_name}: {row['trip_count']:,} trips")

busiest_day = __builtins__.max(daily_results, key=lambda x: x['trip_count'])
print(f"   Busiest day: {day_names[busiest_day['day_of_week']]} ({busiest_day['trip_count']:,} trips)")

# 4. Average tip percentage per payment type
print("\n4. Average tip percentage by payment type:")
tip_by_payment = df_clean.groupBy("payment_type").agg(
    avg("tip_percentage").alias("avg_tip_pct"),
    count("*").alias("trip_count")
).orderBy(desc("avg_tip_pct"))

tip_by_payment.show(truncate=False)

=== 2.5 Aggregations ===

1. Average trip distance and fare by hour of day:
+-----------+------------------+------------------+
|hour_of_day|avg_distance      |avg_fare          |
+-----------+------------------+------------------+
|0          |3.9940352209452854|20.30939805135777 |
|1          |3.451589763554973 |18.065711786363174|
|2          |3.093273722564794 |16.579505236369506|
|3          |3.36976697139736  |17.58139394102555 |
|4          |4.964229166791558 |23.816265099242834|
|5          |6.338921588625854 |28.514970320356465|
|6          |4.942863689279742 |23.142339296536758|
|7          |3.743502452714594 |19.53600729558587 |
|8          |3.224630307714529 |18.576205212677593|
|9          |3.1320892408323777|18.649960359949574|
|10         |3.1565001066716376|19.039094384865432|
|11         |3.130636535000449 |19.304061133703566|
|12         |3.221441404134355 |19.73083647909722 |
|13         |3.412610298999755 |20.460779123627834|
|14         |3.56983635504181  |21.25179

### 2.6 Write Cleaned Data

In [41]:
print("=== 2.6 Write Cleaned Data ===\n")

# Write as single Parquet file
output_path = "/app/data/nyc_taxi_2024_clean.parquet"
print(f"Writing to {output_path}...")

df_clean.coalesce(1).write.mode("overwrite").parquet(output_path)

print(f" Cleaned data written successfully")
print(f"  Location: {output_path}")
print(f"  Rows: {df_clean.count():,}")
print(f"  Columns: {len(df_clean.columns)}")

=== 2.6 Write Cleaned Data ===

Writing to /app/data/nyc_taxi_2024_clean.parquet...
 Cleaned data written successfully
  Location: /app/data/nyc_taxi_2024_clean.parquet
  Rows: 35,623,564
  Columns: 25


## Part 3: Spark SQL & Window Functions (35%)

Using Spark SQL for advanced analytics with window functions.

In [42]:
print("=== 3.0 Setup: Register Temporary View ===\n")

# Load cleaned data and register as temporary view
df_clean_reload = spark.read.parquet("/app/data/nyc_taxi_2024_clean.parquet")
df_clean_reload.createOrReplaceTempView("trips")

print(f"  Temporary view 'trips' registered")
print(f"  Rows: {df_clean_reload.count():,}")

=== 3.0 Setup: Register Temporary View ===

  Temporary view 'trips' registered
  Rows: 35,623,564


### 3.1 SQL Basics

In [43]:
print("=== 3.1 SQL Basics ===\n")

# 3.1.1 Top 10 busiest pickup locations
print("1. Top 10 busiest pickup locations by trip count:")
spark.sql("""
SELECT
    PULocationID,
    COUNT(*) as trip_count
FROM trips
GROUP BY PULocationID
ORDER BY trip_count DESC
LIMIT 10
""").show()

# 3.1.2 Average fare, tip, and total by payment type
print("\n2. Average amounts by payment type:")
spark.sql("""
SELECT
    payment_type,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    ROUND(AVG(tip_amount), 2) as avg_tip,
    ROUND(AVG(total_amount), 2) as avg_total,
    COUNT(*) as trip_count
FROM trips
GROUP BY payment_type
ORDER BY payment_type
""").show()

# 3.1.3 CTE: locations with above-average trip distance
print("\n3. Pickup locations with avg distance > 2x overall average:")
spark.sql("""
WITH overall_avg AS (
    SELECT AVG(trip_distance) as avg_distance FROM trips
),
location_avg AS (
    SELECT
        PULocationID,
        AVG(trip_distance) as avg_distance,
        COUNT(*) as trip_count
    FROM trips
    GROUP BY PULocationID
)
SELECT
    l.PULocationID,
    l.avg_distance,
    l.trip_count
FROM location_avg l, overall_avg o
WHERE l.avg_distance > 2 * o.avg_distance
ORDER BY l.avg_distance DESC
LIMIT 10
""").show()

=== 3.1 SQL Basics ===

1. Top 10 busiest pickup locations by trip count:
+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         132|   1856382|
|         237|   1766762|
|         161|   1737647|
|         236|   1557671|
|         162|   1309910|
|         186|   1269313|
|         230|   1250688|
|         138|   1245356|
|         142|   1182683|
|         163|   1063249|
+------------+----------+


2. Average amounts by payment type:
+------------+--------+-------+---------+----------+
|payment_type|avg_fare|avg_tip|avg_total|trip_count|
+------------+--------+-------+---------+----------+
|           1|   19.75|   4.36|    29.75|  29870931|
|           2|   19.64|    0.0|    24.96|   5220122|
|           3|   19.94|   0.01|    25.27|    148514|
|           4|   22.26|   0.01|    27.91|    383997|
+------------+--------+-------+---------+----------+


3. Pickup locations with avg distance > 2x overall average:
+------------+------------------+-----

### 3.2 CASE Expressions

In [44]:
print("=== 3.2 CASE Expressions ===\n")

# Classify trips into time_of_day and compute statistics
print("Average trip metrics by time of day:")
result = spark.sql("""
SELECT
    CASE
        WHEN hour_of_day >= 6 AND hour_of_day < 12 THEN 'morning'
        WHEN hour_of_day >= 12 AND hour_of_day < 18 THEN 'afternoon'
        WHEN hour_of_day >= 18 AND hour_of_day < 23 THEN 'evening'
        ELSE 'night'
    END as time_of_day,
    ROUND(AVG(trip_distance), 2) as avg_distance,
    ROUND(AVG(total_amount), 2) as avg_total,
    ROUND(AVG(tip_percentage), 2) as avg_tip_pct,
    COUNT(*) as trip_count
FROM trips
GROUP BY
    CASE
        WHEN hour_of_day >= 6 AND hour_of_day < 12 THEN 'morning'
        WHEN hour_of_day >= 12 AND hour_of_day < 18 THEN 'afternoon'
        WHEN hour_of_day >= 18 AND hour_of_day < 23 THEN 'evening'
        ELSE 'night'
    END
ORDER BY
    CASE time_of_day
        WHEN 'morning' THEN 1
        WHEN 'afternoon' THEN 2
        WHEN 'evening' THEN 3
        WHEN 'night' THEN 4
    END
""")
result.show()

most_expensive = result.orderBy(desc("avg_total")).first()
print(f"\nMost expensive trips happen during: {most_expensive['time_of_day']} (${most_expensive['avg_total']:.2f})")

=== 3.2 CASE Expressions ===

Average trip metrics by time of day:
+-----------+------------+---------+-----------+----------+
|time_of_day|avg_distance|avg_total|avg_tip_pct|trip_count|
+-----------+------------+---------+-----------+----------+
|    morning|        3.34|    27.48|      19.92|   7666634|
|  afternoon|        3.42|     30.0|      20.35|  13168208|
|    evening|        3.34|    28.72|      22.55|  10782325|
|      night|        3.96|    29.49|      21.45|   4006397|
+-----------+------------+---------+-----------+----------+


Most expensive trips happen during: afternoon ($30.00)


### 3.3 Window Functions: Ranking

In [45]:
print("=== 3.3 Window Functions: Ranking ===\n")

# 3.3.1 Top 5 locations by revenue per month
print("1. Top 5 pickup locations by revenue per month:")
spark.sql("""
WITH ranked_locations AS (
    SELECT
        month,
        PULocationID,
        SUM(total_amount) as revenue,
        ROW_NUMBER() OVER (PARTITION BY month ORDER BY SUM(total_amount) DESC) as rank
    FROM trips
    GROUP BY month, PULocationID
)
SELECT
    month,
    PULocationID,
    ROUND(revenue, 2) as revenue,
    rank
FROM ranked_locations
WHERE rank <= 5
ORDER BY month, rank
""").show(60, truncate=False)

# 3.3.2 Revenue quartiles per month using NTILE
print("\n2. Revenue quartiles per month (NTILE):")
quartile_result = spark.sql("""
WITH location_revenue AS (
    SELECT
        month,
        PULocationID,
        SUM(total_amount) as total_revenue,
        NTILE(4) OVER (PARTITION BY month ORDER BY SUM(total_amount) DESC) as revenue_quartile
    FROM trips
    GROUP BY month, PULocationID
)
SELECT
    month,
    revenue_quartile,
    COUNT(*) as location_count,
    ROUND(MIN(total_revenue), 2) as min_revenue,
    ROUND(MAX(total_revenue), 2) as max_revenue
FROM location_revenue
GROUP BY month, revenue_quartile
ORDER BY month, revenue_quartile
""")
quartile_result.show()

top_quartile = spark.sql("""
WITH location_revenue AS (
    SELECT
        month,
        PULocationID,
        SUM(total_amount) as total_revenue,
        NTILE(4) OVER (PARTITION BY month ORDER BY SUM(total_amount) DESC) as revenue_quartile
    FROM trips
    GROUP BY month, PULocationID
)
SELECT COUNT(DISTINCT PULocationID) as top_quartile_count
FROM location_revenue
WHERE revenue_quartile = 1
""").first()

print(f"\nTotal locations in top quartile (all months): {top_quartile['top_quartile_count']}")

=== 3.3 Window Functions: Ranking ===

1. Top 5 pickup locations by revenue per month:
+-----+------------+-------------+----+
|month|PULocationID|revenue      |rank|
+-----+------------+-------------+----+
|1    |132         |1.107120876E7|1   |
|1    |138         |5745056.44   |2   |
|1    |161         |3234330.9    |3   |
|1    |237         |2672667.11   |4   |
|1    |230         |2657711.0    |5   |
|2    |132         |9743008.95   |1   |
|2    |138         |5644524.37   |2   |
|2    |161         |3345928.0    |3   |
|2    |237         |2666334.81   |4   |
|2    |230         |2562807.57   |5   |
|3    |132         |1.231711663E7|1   |
|3    |138         |7040654.95   |2   |
|3    |161         |3641500.23   |3   |
|3    |230         |3056386.32   |4   |
|3    |237         |2913362.74   |5   |
|4    |132         |1.222248976E7|1   |
|4    |138         |7297918.37   |2   |
|4    |161         |3682418.4    |3   |
|4    |237         |3078475.67   |4   |
|4    |230         |2982296.61   

### 3.4 Window Functions: Running Totals and Moving Averages

In [46]:
print("=== 3.4 Window Functions: Running Totals & Moving Averages ===\n")

# Complex query with daily revenue, running total, moving average, and lag
print("Daily revenue analysis (first 30 days):")
daily_analysis = spark.sql("""
WITH daily_revenue AS (
    SELECT
        CAST(tpep_pickup_datetime AS DATE) as trip_date,
        SUM(total_amount) as daily_revenue
    FROM trips
    GROUP BY CAST(tpep_pickup_datetime AS DATE)
)
SELECT
    trip_date,
    ROUND(daily_revenue, 2) as daily_revenue,
    ROUND(SUM(daily_revenue) OVER (ORDER BY trip_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) as running_total,
    ROUND(AVG(daily_revenue) OVER (ORDER BY trip_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) as moving_avg_7day,
    ROUND(daily_revenue - LAG(daily_revenue, 1) OVER (ORDER BY trip_date), 2) as day_over_day_change
FROM daily_revenue
ORDER BY trip_date
LIMIT 30
""")
daily_analysis.show(30, truncate=False)

# Find biggest single-day revenue drop
print("\nBiggest single-day revenue drops:")
max_drop = spark.sql("""
WITH daily_revenue AS (
    SELECT
        CAST(tpep_pickup_datetime AS DATE) as trip_date,
        SUM(total_amount) as daily_revenue
    FROM trips
    GROUP BY CAST(tpep_pickup_datetime AS DATE)
),
with_lag AS (
    SELECT
        trip_date,
        daily_revenue,
        LAG(daily_revenue, 1) OVER (ORDER BY trip_date) as prev_revenue,
        daily_revenue - LAG(daily_revenue, 1) OVER (ORDER BY trip_date) as day_over_day_change
    FROM daily_revenue
)
SELECT
    trip_date,
    ROUND(daily_revenue, 2) as daily_revenue,
    ROUND(prev_revenue, 2) as prev_revenue,
    ROUND(day_over_day_change, 2) as change
FROM with_lag
WHERE day_over_day_change IS NOT NULL
ORDER BY day_over_day_change ASC
LIMIT 5
""")
max_drop.show()

=== 3.4 Window Functions: Running Totals & Moving Averages ===

Daily revenue analysis (first 30 days):
+----------+-------------+-------------+---------------+-------------------+
|trip_date |daily_revenue|running_total|moving_avg_7day|day_over_day_change|
+----------+-------------+-------------+---------------+-------------------+
|2024-01-01|2102786.4    |2102786.4    |2102786.4      |NULL               |
|2024-01-02|2179962.09   |4282748.49   |2141374.24     |77175.69           |
|2024-01-03|2265092.25   |6547840.74   |2182613.58     |85130.16           |
|2024-01-04|2687180.78   |9235021.52   |2308755.38     |422088.53          |
|2024-01-05|2600641.59   |1.183566311E7|2367132.62     |-86539.19          |
|2024-01-06|2279012.19   |1.41146753E7 |2352445.88     |-321629.4          |
|2024-01-07|1799281.66   |1.591395696E7|2273422.42     |-479730.53         |
|2024-01-08|2123706.39   |1.803766335E7|2276410.99     |324424.73          |
|2024-01-09|2161180.85   |2.01988442E7 |2273727.9

### 3.5 Window Functions: Comparisons

In [47]:
print("=== 3.5 Window Functions: Comparisons ===\n")

# 3.5.1 Trips vs hourly average
print("1. Trips with highest deviation from hourly average (top 20):")
spark.sql("""
SELECT * FROM (
    SELECT
        hour_of_day,
        ROUND(total_amount, 2) as trip_amount,
        ROUND(AVG(total_amount) OVER (PARTITION BY hour_of_day), 2) as hourly_avg,
        ROUND(total_amount - AVG(total_amount) OVER (PARTITION BY hour_of_day), 2) as deviation
    FROM trips
) t
WHERE deviation IS NOT NULL
ORDER BY deviation DESC
LIMIT 20
""").show(truncate=False)

# 3.5.2 Monthly extremes using FIRST_VALUE and LAST_VALUE
print("\n2. Highest and lowest revenue days per month:")
spark.sql("""
WITH daily_revenue AS (
    SELECT
        month,
        CAST(tpep_pickup_datetime AS DATE) as trip_date,
        SUM(total_amount) as daily_revenue
    FROM trips
    GROUP BY month, CAST(tpep_pickup_datetime AS DATE)
),
monthly_extremes AS (
    SELECT
        month,
        trip_date,
        daily_revenue,
        FIRST_VALUE(trip_date) OVER (PARTITION BY month ORDER BY daily_revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as highest_revenue_date,
        FIRST_VALUE(daily_revenue) OVER (PARTITION BY month ORDER BY daily_revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as highest_revenue,
        FIRST_VALUE(trip_date) OVER (PARTITION BY month ORDER BY daily_revenue ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as lowest_revenue_date,
        FIRST_VALUE(daily_revenue) OVER (PARTITION BY month ORDER BY daily_revenue ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as lowest_revenue
    FROM daily_revenue
)
SELECT DISTINCT
    month,
    highest_revenue_date,
    ROUND(highest_revenue, 2) as highest_revenue,
    lowest_revenue_date,
    ROUND(lowest_revenue, 2) as lowest_revenue
FROM monthly_extremes
ORDER BY month
""").show(truncate=False)

=== 3.5 Window Functions: Comparisons ===

1. Trips with highest deviation from hourly average (top 20):
+-----------+-----------+----------+---------+
|hour_of_day|trip_amount|hourly_avg|deviation|
+-----------+-----------+----------+---------+
|14         |3037.1     |30.1      |3007.0   |
|20         |2265.45    |28.45     |2237.0   |
|10         |2225.3     |27.32     |2197.98  |
|16         |1737.18    |32.08     |1705.1   |
|9          |1477.68    |26.8      |1450.88  |
|18         |1315.97    |28.69     |1287.28  |
|17         |1218.99    |30.21     |1188.78  |
|18         |1204.41    |28.69     |1175.72  |
|23         |1199.67    |30.75     |1168.92  |
|0          |1150.67    |29.69     |1120.98  |
|22         |1111.54    |29.14     |1082.4   |
|23         |1078.87    |30.75     |1048.12  |
|10         |1067.58    |27.32     |1040.26  |
|22         |1021.99    |29.14     |992.85   |
|23         |1018.97    |30.75     |988.22   |
|20         |1001.5     |28.45     |973.05   |
|1

### 3.6 ROLLUP

Hierarchical revenue analysis by month and payment type.

In [48]:
print("=== 3.6 ROLLUP ===\n")

print("Revenue with subtotals by month and payment_type:")
rollup_result = spark.sql("""
SELECT
    month,
    payment_type,
    SUM(total_amount) as total_revenue,
    COUNT(*) as trip_count
FROM trips
GROUP BY ROLLUP(month, payment_type)
ORDER BY month, payment_type
""")
rollup_result.show(100, truncate=False)

print("\nExplanation of NULL rows in ROLLUP:")
print("  - Row with payment_type=NULL: Subtotal for each month (all payment types combined)")
print("  - Row with both month=NULL and payment_type=NULL: Grand total (all data)")

# Count NULL rows
null_rows = spark.sql("""
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN month IS NULL THEN 1 ELSE 0 END) as month_null_count,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) as payment_type_null_count,
    SUM(CASE WHEN month IS NULL AND payment_type IS NULL THEN 1 ELSE 0 END) as both_null_count
FROM (
    SELECT
        month,
        payment_type,
        SUM(total_amount) as total_revenue,
        COUNT(*) as trip_count
    FROM trips
    GROUP BY ROLLUP(month, payment_type)
)
""").collect()[0]

print(f"\nRollup statistics:")
print(f"  Total rows in ROLLUP result: {null_rows['total_rows']}")
print(f"  Month=NULL rows (payment type subtotals): {null_rows['payment_type_null_count']}")
print(f"  Both NULL (grand total): {null_rows['both_null_count']}")

=== 3.6 ROLLUP ===

Revenue with subtotals by month and payment_type:
+-----+------------+--------------------+----------+
|month|payment_type|total_revenue       |trip_count|
+-----+------------+--------------------+----------+
|NULL |NULL        |1.0335206038675189E9|35623564  |
|1    |NULL        |7.461821099003378E7 |2723184   |
|1    |1           |6.385326936004026E7 |2273105   |
|1    |2           |9979769.850000672   |417798    |
|1    |3           |217962.16000000076  |9697      |
|1    |4           |567209.6200000049   |22584     |
|2    |NULL        |7.444384533994398E7 |2719133   |
|2    |1           |6.434241821999472E7 |2294733   |
|2    |2           |9293911.060000887   |391648    |
|2    |3           |221516.05000000112  |9603      |
|2    |4           |586000.009999999    |23149     |
|3    |NULL        |8.601527201001577E7 |3035518   |
|3    |1           |7.401644405003661E7 |2543867   |
|3    |2           |1.0960267690001667E7|451351    |
|3    |3           |282517.24

## Part 4: Performance Analysis (10%)

Query plan analysis, caching effectiveness, and partition pruning.

### 4.1 Query Plans

In [49]:
print("=== 4.1 Query Plans ===\n")

# Define the daily revenue query with running total and moving average
daily_revenue_query = spark.sql("""
WITH daily_revenue AS (
    SELECT
        CAST(tpep_pickup_datetime AS DATE) as trip_date,
        SUM(total_amount) as daily_revenue
    FROM trips
    GROUP BY CAST(tpep_pickup_datetime AS DATE)
)
SELECT
    trip_date,
    daily_revenue,
    SUM(daily_revenue) OVER (ORDER BY trip_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total,
    AVG(daily_revenue) OVER (ORDER BY trip_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as moving_avg_7day
FROM daily_revenue
""")

print("Physical Plan:")
print("=" * 80)
daily_revenue_query.explain(extended=True)
print("=" * 80)

print("\n\nAnalysis:")
print("-" * 80)
print("4.1.1 Physical Plan Structure:")
print("  - Multiple stages visible in the execution tree")
print("  - Window functions trigger shuffles for ordering")
print("  - The running total and moving average require full window computation")

print("\n4.1.2 Shuffle Operations:")
print("  - GROUP BY operation: Creates shuffle to group by trip_date")
print("  - Window functions: Additional shuffle for ORDER BY clause in window")
print("  - Total shuffles: At least 2-3 depending on optimization")

print("\n4.1.3 Column Projection:")
print("  - Only 'tpep_pickup_datetime' and 'total_amount' are read from Parquet")
print("  - Predicate pushdown likely not applicable (aggregate function)")
print("  - Check explain plan for 'PushedFilters' to see if pruning occurs")

=== 4.1 Query Plans ===

Physical Plan:
== Parsed Logical Plan ==
CTE [daily_revenue]
:  +- 'SubqueryAlias daily_revenue
:     +- 'Aggregate [cast('tpep_pickup_datetime as date)], [cast('tpep_pickup_datetime as date) AS trip_date#15109, 'SUM('total_amount) AS daily_revenue#15110]
:        +- 'UnresolvedRelation [trips], [], false
+- 'Project ['trip_date, 'daily_revenue, 'SUM('daily_revenue) windowspecdefinition('trip_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS running_total#15107, 'AVG('daily_revenue) windowspecdefinition('trip_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())) AS moving_avg_7day#15108]
   +- 'UnresolvedRelation [daily_revenue], [], false

== Analyzed Logical Plan ==
trip_date: date, daily_revenue: double, running_total: double, moving_avg_7day: double
WithCTE
:- CTERelationDef 21, false
:  +- SubqueryAlias daily_revenue
:     +- Aggregate [cast(tpep_pickup_datetime#4011 as date)], [cast(tpep_pick

### 4.2 Caching

Measure query execution time with and without caching.

In [50]:
import time

print("=== 4.2 Caching ===\n")

# Reload clean data
df_for_cache = spark.read.parquet("/app/data/nyc_taxi_2024_clean.parquet")
df_for_cache.createOrReplaceTempView("trips_cache")

# Define aggregation query that we'll run multiple times
agg_query = """
SELECT
    month,
    payment_type,
    SUM(total_amount) as total_revenue,
    COUNT(*) as trip_count
FROM trips_cache
GROUP BY month, payment_type
"""

# Test 1: Without caching - run 3 times
print("1. WITHOUT CACHING - 3 runs:")
times_no_cache = []
for i in range(3):
    start = time.time()
    result = spark.sql(agg_query)
    result.collect()  # Force execution
    elapsed = time.time() - start
    times_no_cache.append(elapsed)
    print(f"   Run {i+1}: {elapsed:.3f}s")

avg_no_cache = __builtins__.sum(times_no_cache) / len(times_no_cache)
print(f"   Average: {avg_no_cache:.3f}s\n")

# Test 2: With caching - cache the base table first
print("2. WITH CACHING - cache base table first:")
df_for_cache.cache()
df_for_cache.count()  # Force cache materialization
print("   Base table cached\n")

times_with_cache = []
for i in range(3):
    start = time.time()
    result = spark.sql(agg_query)
    result.collect()  # Force execution
    elapsed = time.time() - start
    times_with_cache.append(elapsed)
    print(f"   Run {i+1}: {elapsed:.3f}s")

avg_with_cache = __builtins__.sum(times_with_cache) / len(times_with_cache)
print(f"   Average: {avg_with_cache:.3f}s\n")

# Comparison
improvement = ((avg_no_cache - avg_with_cache) / avg_no_cache) * 100
print(f"Performance Comparison:")
print(f"  Without cache: {avg_no_cache:.3f}s")
print(f"  With cache:    {avg_with_cache:.3f}s")
print(f"  Improvement:   {improvement:.1f}%")

if improvement < 5:
    print(f"\nObservation: Caching provided minimal improvement (< 5%)")
    print(f"Reasons:")
    print(f"  - Parquet files are already highly optimized")
    print(f"  - Aggregation query may still shuffle despite caching")
    print(f"  - Modern CPUs and SSDs read sequential data very quickly")
else:
    print(f"\nObservation: Caching provided significant speedup!")

# Unpersist cache
df_for_cache.unpersist()

=== 4.2 Caching ===

1. WITHOUT CACHING - 3 runs:
   Run 1: 14.072s
   Run 2: 9.372s
   Run 3: 9.690s
   Average: 11.045s

2. WITH CACHING - cache base table first:
   Base table cached

   Run 1: 11.445s
   Run 2: 11.275s
   Run 3: 9.158s
   Average: 10.626s

Performance Comparison:
  Without cache: 11.045s
  With cache:    10.626s
  Improvement:   3.8%

Observation: Caching provided minimal improvement (< 5%)
Reasons:
  - Parquet files are already highly optimized
  - Aggregation query may still shuffle despite caching
  - Modern CPUs and SSDs read sequential data very quickly


DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, trip_duration_minutes: double, hour_of_day: int, day_of_week: int, month: int, tip_percentage: double, speed_mph: double]

In the case of cache, the data is fully materialized with all columns. Then, the query is executed processing all columns. Therefore, the parquet with column pruning is not efective. That is the reason why the cache case does not imply an improvement.

### 4.3 Partitioned Reads

Compare single file vs partitioned dataset with partition pruning.

In [51]:
print("=== 4.3 Partitioned Reads ===\n")

import pyspark.sql.functions as F

# Write partitioned version by month
partitioned_path = "/app/data/nyc_taxi_2024_partitioned"
print(f"Writing partitioned data to {partitioned_path}...")

df_for_cache.write.mode("overwrite").partitionBy("month").parquet(partitioned_path)
print(" Partitioned data written\n")

# Test 1: Read from single file + filter in memory
print("1. Reading from SINGLE FILE + filter:")
times_single = []
for i in range(3):
    start = time.time()
    df_single = spark.read.parquet("/app/data/nyc_taxi_2024_clean.parquet")
    result = df_single.filter(col("month") == 6)
    result.collect()
    elapsed = time.time() - start
    times_single.append(elapsed)
    print(f"   Run {i+1}: {elapsed:.3f}s")

avg_single = __builtins__.sum(times_single) / len(times_single)
print(f"   Average: {avg_single:.3f}s\n")

# Test 2: Read from the ROOT of partitioned data
print("2. Reading from PARTITIONED data (partition pruning):")
times_partitioned = []
for i in range(3):
    start = time.time()

    # 1. Point to the ROOT directory, not the subfolder
    df_part = spark.read.parquet(partitioned_path)

    # 2. Spark "discovers" the month column from the folder names
    # 3. This filter tells Spark to ONLY read the month=6 folder
    result = df_part.filter(F.col("month") == 6)

    result.collect()
    elapsed = time.time() - start
    times_partitioned.append(elapsed)
    print(f"   Run {i+1}: {elapsed:.3f}s")

# Use __builtins__.sum to avoid the shadowing error
avg_partitioned = __builtins__.sum(times_partitioned) / len(times_partitioned)
print(f"   Average: {avg_partitioned:.3f}s\n")

# Comparison
improvement_pct = ((avg_single - avg_partitioned) / avg_single) * 100
print(f"Performance Comparison:")
print(f"  Single file + filter:     {avg_single:.3f}s")
print(f"  Partitioned (pruned):     {avg_partitioned:.3f}s")
print(f"  Improvement:              {improvement_pct:.1f}%")

if improvement_pct > 10:
    print(f"\n  Significant improvement with partition pruning.")
else:
    print(f"\nNote: Improvement may be limited due to:")
    print(f"  - Small dataset relative to memory")
    print(f"  - Multiple partitions in same query execution")
    print(f"  - File system caching effects")

# Verify partition pruning in explain plan
print(f"\nExplain plan for partitioned read:")
df_part_explain = spark.read.parquet(partitioned_path).filter(F.col("month") == 6)
df_part_explain.explain()

=== 4.3 Partitioned Reads ===

Writing partitioned data to /app/data/nyc_taxi_2024_partitioned...
 Partitioned data written

1. Reading from SINGLE FILE + filter:
   Run 1: 79.013s
   Run 2: 82.055s
   Run 3: 77.425s
   Average: 79.498s

2. Reading from PARTITIONED data (partition pruning):
   Run 1: 79.622s
   Run 2: 78.138s
   Run 3: 75.645s
   Average: 77.802s

Performance Comparison:
  Single file + filter:     79.498s
  Partitioned (pruned):     77.802s
  Improvement:              2.1%

Note: Improvement may be limited due to:
  - Small dataset relative to memory
  - Multiple partitions in same query execution
  - File system caching effects

Explain plan for partitioned read:
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#20144,tpep_pickup_datetime#20145,tpep_dropoff_datetime#20146,passenger_count#20147L,trip_distance#20148,RatecodeID#20149L,store_and_fwd_flag#20150,PULocationID#20151,DOLocationID#20152,payment_type#20153L,fare_amount#20154,extra#20155,mta_t